In [1]:
# Comprehensive marker testing for poolparty
import poolparty as pp
from poolparty.markers import (
    find_all_markers,
    has_marker,
    validate_single_marker,
    strip_all_markers,
    get_length_without_markers,
    get_positions_without_markers,
    build_marker_tag,
)

In [2]:
# --- Parsing: find_all_markers with various marker types ---
# Zero-length marker
m1 = find_all_markers('ACGT<ins/>TT')
assert len(m1) == 1 and m1[0].name == 'ins' and m1[0].content == '' and m1[0].is_zero_length

# Region marker with content
m2 = find_all_markers('AC<region>TG</region>AA')
assert len(m2) == 1 and m2[0].name == 'region' and m2[0].content == 'TG'

# Strand attribute
m3 = find_all_markers("AC<region strand='-'>TG</region>AA")
assert m3[0].strand == '-'

# seq_length attribute (integer)
m4 = find_all_markers("<orf seq_length='4'>ACGT</orf>")
assert m4[0].declared_seq_length == 4

# seq_length='None' for variable length
m5 = find_all_markers("<var seq_length='None'>ACGT</var>")
assert m5[0].declared_seq_length == -1 and m5[0].is_variable_length

# Multiple markers
m6 = find_all_markers('A<m1>B</m1>C<m2/>D')
assert len(m6) == 2 and m6[0].name == 'm1' and m6[1].name == 'm2'

# has_marker and validate_single_marker
assert has_marker('AC<region>TG</region>AA', 'region') == True
assert has_marker('ACGT', 'region') == False
vm = validate_single_marker('AC<region>TG</region>AA', 'region')
assert vm.name == 'region' and vm.content == 'TG'

,seq,pool[1].seq,pool[0].seq,pool[1].state,pool[0].state,op[1]:fixed.state,op[0]:fixed.state
0,ACGT<insert/>ACGTACGT,ACGT<insert/>ACGTACGT,ACGTACGTACGT,0,0,0,0


In [3]:
# --- Parsing: string manipulation utilities ---
# strip_all_markers removes tags but keeps content
assert strip_all_markers('AC<region>TG</region>AA') == 'ACTGAA'
assert strip_all_markers('AC<ins/>TG') == 'ACTG'
assert strip_all_markers('A<m1>B</m1>C<m2/>D') == 'ABCD'

# get_length_without_markers returns sequence length excluding tags
assert get_length_without_markers('ACGT') == 4
assert get_length_without_markers('AC<region>TG</region>AA') == 6
assert get_length_without_markers('AC<ins/>GT') == 4

# get_positions_without_markers returns raw string positions of non-tag characters
assert get_positions_without_markers('ACGT') == [0, 1, 2, 3]
pos = get_positions_without_markers('AC<ins/>GT')  # A=0, C=1, <ins/>=2-7, G=8, T=9
assert pos == [0, 1, 8, 9]

# build_marker_tag constructs marker strings
assert build_marker_tag('ins', '') == '<ins/>'
assert build_marker_tag('ins', '', '-') == "<ins strand='-'/>"
assert build_marker_tag('region', 'ACGT') == '<region>ACGT</region>'
assert build_marker_tag('region', 'ACGT', '-') == "<region strand='-'>ACGT</region>"
assert build_marker_tag('orf', 'ACGT', '+', seq_length=4) == "<orf seq_length='4'>ACGT</orf>"
assert build_marker_tag('var', 'ACGT', '+', seq_length=-1) == "<var seq_length='None'>ACGT</var>"

,seq,pool[1].seq,pool[0].seq,pool[1].state,pool[0].state,op[1]:fixed.state,op[0]:fixed.state
0,AC<region>GTAC</region>GT,AC<region>GTAC</region>GT,ACGTACGT,0,0,0,0


In [4]:
# --- insert_marker: fixed position markers ---
pp.reset_default_party()

# Zero-length marker at position 4
p1 = pp.insert_marker('ACGTACGT', 'ins', start=4)
df1 = p1.generate_seqs(num_seqs=1)
assert df1['seq'].iloc[0] == 'ACGT<ins/>ACGT'

# Region marker encompassing positions 2-5
pp.reset_default_party()
p2 = pp.insert_marker('ACGTACGT', 'region', start=2, stop=5)
df2 = p2.generate_seqs(num_seqs=1)
assert df2['seq'].iloc[0] == 'AC<region>GTA</region>CGT'

# Marker with strand='-'
pp.reset_default_party()
p3 = pp.insert_marker('ACGT', 'site', start=1, stop=3, strand='-')
df3 = p3.generate_seqs(num_seqs=1)
assert df3['seq'].iloc[0] == "A<site strand='-'>CG</site>T"

,seq,pool[1].seq,pool[0].seq,pool[1].state,pool[0].state,op[1]:marker_scan.state,op[0]:fixed.state,op[1]:marker_scan.key.position,op[1]:marker_scan.key.strand,op[1]:marker_scan.key.marker_tag
0,<m/>ACGT,<m/>ACGT,ACGT,0,0,0,0,0,+,<m/>
1,A<m/>CGT,A<m/>CGT,ACGT,1,0,1,0,1,+,<m/>
2,AC<m/>GT,AC<m/>GT,ACGT,2,0,2,0,2,+,<m/>
3,ACG<m/>T,ACG<m/>T,ACGT,3,0,3,0,3,+,<m/>
4,ACGT<m/>,ACGT<m/>,ACGT,4,0,4,0,4,+,<m/>


In [5]:
# --- marker_scan: sequential mode ---
pp.reset_default_party()

# Zero-length markers at all positions (5 positions for 4-char seq)
p1 = pp.marker_scan('ACGT', marker='m', mode='sequential')
df1 = p1.generate_seqs(num_complete_iterations=1)
assert len(df1) == 5
seqs1 = set(df1['seq'])
assert '<m/>ACGT' in seqs1 and 'A<m/>CGT' in seqs1 and 'ACGT<m/>' in seqs1

# Region markers with marker_length=2 (3 positions)
pp.reset_default_party()
p2 = pp.marker_scan('ACGT', marker='r', mode='sequential', marker_length=2)
df2 = p2.generate_seqs(num_complete_iterations=1)
assert len(df2) == 3
seqs2 = set(df2['seq'])
assert '<r>AC</r>GT' in seqs2 and 'A<r>CG</r>T' in seqs2 and 'AC<r>GT</r>' in seqs2

# Position filtering - only positions 1 and 3
pp.reset_default_party()
p3 = pp.marker_scan('ACGT', marker='m', mode='sequential', positions=[1, 3])
df3 = p3.generate_seqs(num_complete_iterations=1)
assert len(df3) == 2
seqs3 = set(df3['seq'])
assert 'A<m/>CGT' in seqs3 and 'ACG<m/>T' in seqs3

,seq,pool[1].seq,pool[0].seq,pool[1].state,pool[0].state,op[1]:fixed.state,op[0]:fixed.state
0,CCCC,CCCC,AAAA<target>CCCC</target>TTTT,0,0,0,0


In [6]:
# --- marker_scan: random mode and strand options ---
pp.reset_default_party()

# Random mode - each sequence has marker at random position
p1 = pp.marker_scan('ACGTACGT', marker='m', mode='random')
df1 = p1.generate_seqs(num_seqs=10, seed=42)
for seq in df1['seq']:
    assert '<m/>' in seq

# strand='both' doubles states (5 positions x 2 strands = 10)
pp.reset_default_party()
p2 = pp.marker_scan('ACGT', marker='m', mode='sequential', strand='both')
df2 = p2.generate_seqs(num_complete_iterations=1)
assert len(df2) == 10
# Check both strands appear
plus_count = sum("<m/>" in s and "strand" not in s for s in df2['seq'])
minus_count = sum("strand='-'" in s for s in df2['seq'])
assert plus_count == 5 and minus_count == 5

# strand='-' only
pp.reset_default_party()
p3 = pp.marker_scan('ACGT', marker='m', mode='sequential', strand='-')
df3 = p3.generate_seqs(num_complete_iterations=1)
assert len(df3) == 5
for seq in df3['seq']:
    assert "strand='-'" in seq

,seq,pool[2].seq,pool[0].seq,pool[1].seq,pool[2].state,pool[0].state,pool[1].state,op[2]:replace_marker_content.state,op[0]:fixed.state,op[1]:from_seqs.state,op[1]:from_seqs.key.seq_name,op[1]:from_seqs.key.seq_index
0,AAAAGGGTTTT,AAAAGGGTTTT,AAAA<var/>TTTT,GGG,0,0,0,0,0,0,seq_0,0
1,AAAACCCTTTT,AAAACCCTTTT,AAAA<var/>TTTT,CCC,1,0,1,0,0,1,seq_1,1
2,AAAAAAATTTT,AAAAAAATTTT,AAAA<var/>TTTT,AAA,2,0,2,0,0,2,seq_2,2


In [14]:
# --- marker_multiscan: multiple marker insertion ---
pp.reset_default_party()

# Insert 2 markers with ordered mode (markers[i] -> position[i])
p1 = pp.marker_multiscan('ACGTACGTACGT', markers=['m1', 'm2'], num_insertions=2, mode='random')
df1 = p1.generate_seqs(num_seqs=5, seed=42)
for seq in df1['seq']:
    assert '<m1/>' in seq and '<m2/>' in seq

# With marker_length > 0 for region markers
pp.reset_default_party()
p2 = pp.marker_multiscan('ACGTACGTACGT', markers=['a', 'b'], num_insertions=2, 
                          marker_length=2, mode='random')
df2 = p2.generate_seqs(num_seqs=5, seed=42)
for seq in df2['seq']:
    # Both region markers should be present with 2-char content
    assert '<a>' in seq and '</a>' in seq
    assert '<b>' in seq and '</b>' in seq


,seq,pool[3].seq,pool[2].seq,pool[1].seq,pool[0].seq,pool[3].state,pool[2].state,pool[1].state,pool[0].state,op[3]:replace_marker_content.state,op[2]:mutagenize.state,op[1]:fixed.state,op[0]:fixed.state,op[2]:mutagenize.key.positions,op[2]:mutagenize.key.wt_chars,op[2]:mutagenize.key.mut_chars
0,AAAA.ACCC.TTTT,AAAA.ACCC.TTTT,.ACCC.,.CCCC.,AAAA<target>.CCCC.</target>TTTT,0,0,0,0,0,0,0,0,"(0,)","(C,)","(A,)"
1,AAAA.GCCC.TTTT,AAAA.GCCC.TTTT,.GCCC.,.CCCC.,AAAA<target>.CCCC.</target>TTTT,1,1,0,0,0,1,0,0,"(0,)","(C,)","(G,)"
2,AAAA.TCCC.TTTT,AAAA.TCCC.TTTT,.TCCC.,.CCCC.,AAAA<target>.CCCC.</target>TTTT,2,2,0,0,0,2,0,0,"(0,)","(C,)","(T,)"
3,AAAA.CACC.TTTT,AAAA.CACC.TTTT,.CACC.,.CCCC.,AAAA<target>.CCCC.</target>TTTT,3,3,0,0,0,3,0,0,"(1,)","(C,)","(A,)"
4,AAAA.CGCC.TTTT,AAAA.CGCC.TTTT,.CGCC.,.CCCC.,AAAA<target>.CCCC.</target>TTTT,4,4,0,0,0,4,0,0,"(1,)","(C,)","(G,)"
5,AAAA.CTCC.TTTT,AAAA.CTCC.TTTT,.CTCC.,.CCCC.,AAAA<target>.CCCC.</target>TTTT,5,5,0,0,0,5,0,0,"(1,)","(C,)","(T,)"
6,AAAA.CCAC.TTTT,AAAA.CCAC.TTTT,.CCAC.,.CCCC.,AAAA<target>.CCCC.</target>TTTT,6,6,0,0,0,6,0,0,"(2,)","(C,)","(A,)"
7,AAAA.CCGC.TTTT,AAAA.CCGC.TTTT,.CCGC.,.CCCC.,AAAA<target>.CCCC.</target>TTTT,7,7,0,0,0,7,0,0,"(2,)","(C,)","(G,)"
8,AAAA.CCTC.TTTT,AAAA.CCTC.TTTT,.CCTC.,.CCCC.,AAAA<target>.CCCC.</target>TTTT,8,8,0,0,0,8,0,0,"(2,)","(C,)","(T,)"
9,AAAA.CCCA.TTTT,AAAA.CCCA.TTTT,.CCCA.,.CCCC.,AAAA<target>.CCCC.</target>TTTT,9,9,0,0,0,9,0,0,"(3,)","(C,)","(A,)"


In [9]:
# --- extract_marker_content ---
pp.reset_default_party()

# Basic extraction
bg1 = pp.from_seq('ACGT<region>TTAA</region>GCGC')
content1 = pp.extract_marker_content(bg1, 'region')
df1 = content1.generate_seqs(num_seqs=1)
assert df1['seq'].iloc[0] == 'TTAA'

# Extraction with strand='-' reverse complements the content
pp.reset_default_party()
bg2 = pp.from_seq("ACGT<region strand='-'>AACG</region>GCGC")
content2 = pp.extract_marker_content(bg2, 'region')
df2 = content2.generate_seqs(num_seqs=1)
# AACG reverse complement = CGTT
assert df2['seq'].iloc[0] == 'CGTT'

# Extracted content pool has correct seq_length
pp.reset_default_party()
bg3 = pp.from_seq('AAA<orf>ATGCCC</orf>TTT')
content3 = pp.extract_marker_content(bg3, 'orf')
assert content3.seq_length == 6

Signature:
pp.mutagenize(
    pool: Union[poolparty.pool.Pool, str],
    num_mutations: Optional[numbers.Integral] = None,
    mutation_rate: Optional[numbers.Real] = None,
    mark_changes: Optional[bool] = None,
    mode: Literal['random', 'sequential', 'fixed', 'hybrid'] = 'random',
    num_hybrid_states: Optional[int] = None,
    name: Optional[str] = None,
    op_name: Optional[str] = None,
    iter_order: Optional[numbers.Real] = None,
    op_iter_order: Optional[numbers.Real] = None,
) -> poolparty.pool.Pool
Docstring:
Create a Pool that applies mutations to a sequence.

Parameters
----------
pool : Union[Pool, str]
    Parent pool or sequence string to mutate.
num_mutations : Optional[Integral], default=None
    Fixed number of mutations to apply (mutually exclusive with mutation_rate).
mutation_rate : Optional[Real], default=None
    Probability of mutation at each position (mutually exclusive with num_mutations).
mark_changes : Optional[bool], default=None
    If True, apply 

In [ ]:
# --- replace_marker_content ---
pp.reset_default_party()

# Replace zero-length marker with content
bg1 = pp.from_seq('ACGT<insert/>TTTT')
inserts1 = pp.from_seqs(['AAA', 'GGG'], mode='sequential')
result1 = pp.replace_marker_content(bg1, inserts1, 'insert')
df1 = result1.generate_seqs(num_complete_iterations=1)
seqs1 = set(df1['seq'])
assert 'ACGTAAATTTT' in seqs1 and 'ACGTGGGTTTT' in seqs1

# Replace region marker (replaces existing content)
pp.reset_default_party()
bg2 = pp.from_seq('PREFIX<var>OLDCONTENT</var>SUFFIX')
variants = pp.from_seqs(['NEW1', 'NEW2'], mode='sequential')
result2 = pp.replace_marker_content(bg2, variants, 'var')
df2 = result2.generate_seqs(num_complete_iterations=1)
seqs2 = set(df2['seq'])
assert 'PREFIXNEW1SUFFIX' in seqs2 and 'PREFIXNEW2SUFFIX' in seqs2

# Replace with strand='-' reverse complements content before insertion
pp.reset_default_party()
bg3 = pp.from_seq("ACGT<region strand='-'>XX</region>TTTT")
content3 = pp.from_seq('AAA')
result3 = pp.replace_marker_content(bg3, content3, 'region')
df3 = result3.generate_seqs(num_seqs=1)
# AAA reverse complement = TTT
assert df3['seq'].iloc[0] == 'ACGTTTTTTTT'

In [ ]:
# --- apply_at_marker: transform content ---
pp.reset_default_party()

# Apply reverse_complement at marker
bg1 = pp.from_seq('ACGT<orf>ATGCCC</orf>TTTT')
result1 = pp.apply_at_marker(bg1, 'orf', pp.reverse_complement)
df1 = result1.generate_seqs(num_seqs=1)
# ATGCCC reverse complement = GGGCAT
assert df1['seq'].iloc[0] == 'ACGTGGGCATTTTT'

# Apply mutagenize at marker (sequential enumeration)
pp.reset_default_party()
bg2 = pp.from_seq('AAAA<target>ACGT</target>TTTT')
result2 = pp.apply_at_marker(
    bg2, 'target',
    lambda p: pp.mutagenize(p, num_mutations=1, mode='sequential')
)
df2 = result2.generate_seqs(num_complete_iterations=1)
# 4 positions x 3 alternatives = 12 single-mutation variants
assert len(df2) == 12
for seq in df2['seq']:
    assert len(seq) == 12  # AAAA + 4 + TTTT

# Apply seq_shuffle at marker
pp.reset_default_party()
bg3 = pp.from_seq('AAA<region>ACGTACGT</region>TTT')
result3 = pp.apply_at_marker(bg3, 'region', lambda p: pp.seq_shuffle(p, mode='random'))
df3 = result3.generate_seqs(num_seqs=5, seed=42)
for seq in df3['seq']:
    assert seq.startswith('AAA') and seq.endswith('TTT')
    assert len(seq[3:-3]) == 8  # shuffled 8-char region

In [ ]:
# --- remove_marker ---
pp.reset_default_party()

# Remove marker keeping content (default)
bg1 = pp.from_seq('ACGT<region>TTAA</region>GCGC')
result1 = pp.remove_marker(bg1, 'region', keep_content=True)
df1 = result1.generate_seqs(num_seqs=1)
assert df1['seq'].iloc[0] == 'ACGTTTAAGCGC'

# Remove marker discarding content
pp.reset_default_party()
bg2 = pp.from_seq('ACGT<region>TTAA</region>GCGC')
result2 = pp.remove_marker(bg2, 'region', keep_content=False)
df2 = result2.generate_seqs(num_seqs=1)
assert df2['seq'].iloc[0] == 'ACGTGCGC'

# Remove zero-length marker
pp.reset_default_party()
bg3 = pp.from_seq('ACGT<ins/>GCGC')
result3 = pp.remove_marker(bg3, 'ins')
df3 = result3.generate_seqs(num_seqs=1)
assert df3['seq'].iloc[0] == 'ACGTGCGC'

In [ ]:
# --- Marker registration and pool tracking ---
# Auto-registration via from_seq with embedded markers
with pp.Party() as party:
    pool = pp.from_seq('AAA<orf>ATGCCC</orf>TTT')
    assert party.has_marker('orf')
    m = party.get_marker_by_name('orf')
    assert m.seq_length == 6  # inferred from 'ATGCCC'

# Pool.markers and pool.has_marker
with pp.Party():
    pool = pp.from_seq('AAA<orf>ATGCCC</orf>TTT')
    assert pool.has_marker('orf')
    assert not pool.has_marker('nonexistent')
    assert len(pool.markers) == 1
    assert any(m.name == 'orf' for m in pool.markers)

# Marker inheritance to child pools
with pp.Party():
    parent = pp.from_seq('AAA<orf>ATGCCC</orf>TTT')
    child = pp.reverse_complement(parent)
    assert child.has_marker('orf')

# Marker removed after replace_marker_content
with pp.Party():
    bg = pp.from_seq('AAA<target>XXXX</target>TTT')
    assert bg.has_marker('target')
    content = pp.from_seq('GGGG')
    result = pp.replace_marker_content(bg, content, 'target')
    assert not result.has_marker('target')

In [ ]:
# --- Integration: complete workflows ---
# Workflow 1: insert marker -> transform -> result has no marker tags
with pp.Party():
    bg = pp.from_seq('AAAACGTACGTTTTT')  # 15 chars
    marked = pp.insert_marker(bg, 'target', start=4, stop=12)  # marks CGTACGTT
    result = pp.apply_at_marker(marked, 'target', pp.reverse_complement)
    df = result.generate_seqs(num_seqs=1)
    seq = df['seq'].iloc[0]
    assert '<' not in seq  # no marker tags in output
    assert len(seq) == 15

# Workflow 2: marker_scan + replace_marker_content
with pp.Party():
    marked = pp.marker_scan('ACGTACGT', marker='ins', positions=[2, 6], mode='sequential')
    inserts = pp.from_seq('NNN')
    result = pp.replace_marker_content(marked, inserts, 'ins')
    df = result.generate_seqs(num_complete_iterations=1)
    for seq in df['seq']:
        assert 'NNN' in seq
        assert '<' not in seq  # no markers left

# Workflow 3: mutagenize_orf preserves markers
with pp.Party():
    orf_with_marker = 'ATGTGT<site/>GGTTAA'  # ATG TGT GGT TAA
    pool = pp.from_seq(orf_with_marker)
    mutated = pp.mutagenize_orf(pool, num_mutations=1, codon_positions=[1], mode='random')
    df = mutated.generate_seqs(num_seqs=5, seed=42)
    for seq in df['seq']:
        assert '<site/>' in seq  # marker preserved
        clean = strip_all_markers(seq)
        assert clean[:3] == 'ATG' and clean[-3:] == 'TAA'  # start/stop intact